In [1]:
# CELL 1 — Install Dependencies
# Standard LoRA requires only these packages — bitsandbytes removed entirely.
# bitsandbytes is only needed for 4-bit/8-bit quantization (QLoRA).
# For full-precision LoRA we only need: transformers, peft, trl, accelerate.

!pip install transformers datasets peft trl accelerate pandas -q


In [2]:
from flash_attn import flash_attn_func

print("FlashAttention OK")

FlashAttention OK


In [2]:
# ============================================================
#  CELL 2: HARDWARE SETUP — FIXED FOR TURING GPUs
#  Target: Xenon CPU @ 70% + Quadro RTX 8000 (Turing, CC 7.5)
# ============================================================

import os
import multiprocessing

TOTAL_CPU_CORES = multiprocessing.cpu_count()
CPU_TARGET_PCT  = 0.70
NUM_CPU_WORKERS = max(4, int(TOTAL_CPU_CORES * CPU_TARGET_PCT))

os.environ["OMP_NUM_THREADS"]                   = str(NUM_CPU_WORKERS)
os.environ["MKL_NUM_THREADS"]                   = str(NUM_CPU_WORKERS)
os.environ["TOKENIZERS_PARALLELISM"]            = "true"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"]           = "expandable_segments:True"

import torch
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

if not torch.cuda.is_available():
    raise RuntimeError("[ERROR] No GPU detected. Check CUDA installation.")

GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9
GPU_CAP  = torch.cuda.get_device_capability(0)

print(f"[CPU] Total cores          : {TOTAL_CPU_CORES}")
print(f"[CPU] Workers (70%)        : {NUM_CPU_WORKERS}")
print(f"[GPU] Device               : {GPU_NAME}")
print(f"[GPU] VRAM                 : {GPU_VRAM:.1f} GB")
print(f"[GPU] Compute capability   : {GPU_CAP[0]}.{GPU_CAP[1]}")

# Turing (CC 7.5) does NOT support bfloat16 natively
SUPPORTS_BF16 = GPU_CAP[0] >= 8
COMPUTE_DTYPE = torch.bfloat16 if SUPPORTS_BF16 else torch.float16
USE_BF16      = SUPPORTS_BF16
USE_FP16      = not SUPPORTS_BF16

print(f"[PRECISION] Compute dtype  : {'bfloat16' if USE_BF16 else 'float16'}")
print(f"[PRECISION] BF16 training  : {USE_BF16}")
print(f"[PRECISION] FP16 training  : {USE_FP16}")

# ── ATTENTION IMPLEMENTATION: Force SDPA for Turing GPUs ──
# FlashAttention 2 requires Ampere (CC 8.0+) or newer
# Turing (RTX 8000, CC 7.5) must use PyTorch's native SDPA
if GPU_CAP[0] < 8:
    ATTN_IMPL = "sdpa"
    print(f"[ATTN] GPU is Turing (CC {GPU_CAP[0]}.{GPU_CAP[1]}) — FlashAttention 2 NOT supported")
    print(f"[ATTN] Using 'sdpa' (PyTorch native scaled dot-product attention)")
else:
    try:
        import flash_attn
        ATTN_IMPL = "flash_attention_2"
        print("[ATTN] Flash Attention 2 — fused CUDA kernel active")
    except ImportError:
        ATTN_IMPL = "sdpa"
        print("[ATTN] flash-attn not installed — using sdpa")

print(f"[ATTN] Selected implementation: {ATTN_IMPL}")

[CPU] Total cores          : 48
[CPU] Workers (70%)        : 33
[GPU] Device               : Quadro RTX 8000
[GPU] VRAM                 : 50.7 GB
[GPU] Compute capability   : 7.5
[PRECISION] Compute dtype  : float16
[PRECISION] BF16 training  : False
[PRECISION] FP16 training  : True
[ATTN] GPU is Turing (CC 7.5) — FlashAttention 2 NOT supported
[ATTN] Using 'sdpa' (PyTorch native scaled dot-product attention)
[ATTN] Selected implementation: sdpa


In [ ]:
# ============================================================
#  CELL 3: HUGGINGFACE AUTHENTICATION
#  LLaMA 3.1 is gated — requires HF token with Meta license
#  acceptance at huggingface.co/meta-llama
# ============================================================

from huggingface_hub import login

HF_TOKEN = os.environ.get("HF_TOKEN", "")
login(token=HF_TOKEN)
print("[HF] Authentication successful.")

[HF] Authentication successful.


In [4]:
# ============================================================
#  CELL 4: DATASET PREPARATION
# ============================================================

# CELL 4: DATASET PREPARATION FOR QWEN

DATA_PATH = "gurukul_2_v1 (1).zip"   # Update path if needed

df = pd.read_csv(DATA_PATH)
print(f"[DATA] Raw shape: {df.shape}")
print(f"[DATA] Columns  : {list(df.columns)}")

def format_row_qwen(row):
    """
    Formats data into Qwen2.5 ChatML format:
    <|im_start|>system\n{system}\n<|im_end|>\n
    <|im_start|>user\n{user}\n<|im_end|>\n
    <|im_start|>assistant\n{assistant}<|im_end|>\n
    """
    system_prompt = "You are Gurukul, an intelligent educational assistant specialized in NCERT curriculum."
    
    return {
        "text": (
            f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
            f"<|im_start|>user\nContext: {row['Context']}\n\nQuestion: {row['Question']}<|im_end|>\n"
            f"<|im_start|>assistant\n{row['Answer']}<|im_end|>\n"
        )
    }

raw_dataset = Dataset.from_pandas(df)
formatted_dataset = raw_dataset.map(
    format_row_qwen,
    num_proc=NUM_CPU_WORKERS,
    remove_columns=raw_dataset.column_names,
    desc="Formatting dataset for Qwen"
)

dataset = formatted_dataset.train_test_split(test_size=0.05, seed=42)

print(f"\n[DATA] Train samples : {len(dataset['train'])}")
print(f"[DATA] Test samples  : {len(dataset['test'])}")
print(f"[DATA] Sample preview: ")
print(dataset['train'][0]['text'][:300], "...")

Setting TOKENIZERS_PARALLELISM=false for forked processes.


[DATA] Raw shape: (18721, 6)
[DATA] Columns  : ['Question', 'Answer', 'Chapter', 'Context', 'Class', 'Subject']


Formatting dataset for Qwen (num_proc=33):   0%|          | 0/18721 [00:00<?, ? examples/s]


[DATA] Train samples : 17784
[DATA] Test samples  : 937
[DATA] Sample preview: 
<|im_start|>system
You are Gurukul, an intelligent educational assistant specialized in NCERT curriculum.<|im_end|>
<|im_start|>user
Context: Here's a detailed educational context for the Q&A pair:

"In this context, we're exploring the concept of inertia, which states that an object at rest will re ...


In [5]:
lengths = []

for x in dataset["train"].select(range(min(1000, len(dataset["train"])))):
    lengths.append(len(tokenizer(x["text"]).input_ids))

print("Average length:", sum(lengths)/len(lengths))
print("Max length:", max(lengths))

lengths_sorted = sorted(lengths)

print("95th percentile:", lengths_sorted[int(len(lengths_sorted)*0.95)])
print("99th percentile:", lengths_sorted[int(len(lengths_sorted)*0.99)])

NameError: name 'tokenizer' is not defined

In [6]:
# ============================================================
#  CELL 5: MODEL LOADING — QWEN2.5-7B-INSTRUCT (TURING FIX)
# ============================================================

model_id = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    use_fast=True,
    padding_side="right",
)
tokenizer.pad_token = tokenizer.eos_token

# Load model with explicit attention implementation
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=COMPUTE_DTYPE,  # float16 for Turing
    device_map="auto",
    trust_remote_code=True,
    attn_implementation=ATTN_IMPL,  # "sdpa" for Turing
)

# 🔥 CRITICAL: Explicitly override config to prevent FlashAttention fallback
model.config._attn_implementation = ATTN_IMPL
print(f"[CONFIG] _attn_implementation set to: {model.config._attn_implementation}")

# Verify no FlashAttention in config
if "flash" in str(model.config._attn_implementation).lower():
    raise RuntimeError("[ERROR] FlashAttention still in config! Check ATTN_IMPL setting.")

model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)
model.config.use_cache = False

for param in model.parameters():
    param.requires_grad = False

vram_used = torch.cuda.memory_allocated(0) / 1e9
print(f"\n[MODEL] Loaded : {model_id}")
print(f"[VRAM]  After load : {vram_used:.2f} GB / {GPU_VRAM:.1f} GB")
print(f"[INFO]  Model dtype : {next(model.parameters()).dtype}")

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

[CONFIG] _attn_implementation set to: sdpa

[MODEL] Loaded : Qwen/Qwen2.5-7B-Instruct
[VRAM]  After load : 15.23 GB / 50.7 GB
[INFO]  Model dtype : torch.float16


In [7]:
from flash_attn import flash_attn_func
print("FlashAttention Loaded Successfully")

FlashAttention Loaded Successfully


In [8]:
# ============================================================
#  CELL 6: STANDARD LoRA CONFIGURATION
# ============================================================

# CELL 6: STANDARD LoRA CONFIGURATION FOR QWEN

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

vram_after_lora = torch.cuda.memory_allocated(0) / 1e9
print(f"[VRAM] After LoRA injection : {vram_after_lora:.2f} GB / {GPU_VRAM:.1f} GB")

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273
[VRAM] After LoRA injection : 15.39 GB / 50.7 GB


In [ ]:
# ============================================================
#  CELL 7: TRAINING CONFIGURATION — STANDARD LoRA
#
#  KEY DIFFERENCES vs QLoRA version:
#
#  1. per_device_train_batch_size: 16 → 4
#     Full FP16 model uses ~16 GB vs ~5 GB for 4-bit.
#     Activations for batch=4 at length=2048 use ~6-8 GB.
#     Total: ~16 + 0.3 + 2 + 8 = ~26 GB — safe on 48 GB.
#
#  2. gradient_accumulation_steps: 1 → 4
#     Effective batch = 4 x 4 = 16 — same as QLoRA version.
#
#  3. optim: adamw_torch_fused → paged_adamw_32bit
#     32-bit optimizer states, CPU paging enabled.
#
#  4. learning_rate: 1e-4 → 2e-4 (user preference)
#
#  5. max_length: 1024 → 2048 (user preference)
#
#  6. Precision: bf16=True, fp16=False
#
#  NOTE on group_by_length:
#     Removed — transformers >= 4.45 dropped it.
#     Use packing=True or pre-sort dataset instead.
# ============================================================

OUTPUT_DIR = "./gurukul-llama3-finetuned"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",

    # ── Sequence length ──────────────────────────────────────
    max_length=2048,

    # ── Batch & gradient config ───────────────────────────────
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,   # effective batch = 4 x 4 = 16

    # ── Mixed precision ───────────────────────────────────────
    fp16=True,
    bf16=False,

    # ── Training schedule ─────────────────────────────────────
    num_train_epochs=3,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    weight_decay=0.01,

    # ── CPU DataLoader ────────────────────────────────────────
    dataloader_num_workers=NUM_CPU_WORKERS,
    dataloader_pin_memory=True,

    # ── Optimizer ─────────────────────────────────────────────
    optim="adamw_torch_fused",

    # ── Gradient checkpointing ────────────────────────────────
    gradient_checkpointing=True,

    # ── Sequence packing ──────────────────────────────────────
    packing=False,

    # ── torch.compile ─────────────────────────────────────────
    torch_compile=False,

    # ── Logging & checkpointing ───────────────────────────────
    logging_steps=50,
    save_steps=400,
    eval_strategy="steps",
    eval_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    save_total_limit=2,
    report_to="none",
    seed=42,
)

# ── SFTTrainer ────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
    args=sft_config,
)

# ── Pre-training VRAM report ──────────────────────────────────
torch.cuda.empty_cache()
vram_before   = torch.cuda.memory_allocated(0) / 1e9
vram_reserved = torch.cuda.memory_reserved(0) / 1e9
print(f"[VRAM] Allocated  : {vram_before:.2f} GB")
print(f"[VRAM] Reserved   : {vram_reserved:.2f} GB")
print(f"[VRAM] Headroom   : {GPU_VRAM - vram_reserved:.2f} GB")

print("\n" + "="*60)
print("  TRAINING START — STANDARD LoRA (BF16)")
print(f"  Batch size (per device) : {sft_config.per_device_train_batch_size}")
print(f"  Gradient accum steps    : {sft_config.gradient_accumulation_steps}")
print(f"  Effective batch size    : {sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps}")
print(f"  Max sequence length     : {sft_config.max_length}")
print(f"  CPU workers (70%)       : {NUM_CPU_WORKERS} of {TOTAL_CPU_CORES} cores")
print(f"  Precision               : BF16")
print(f"  Optimizer               : {sft_config.optim}")
print(f"  Flash Attention         : {ATTN_IMPL}")
print("="*60)

trainer.train()

vram_peak = torch.cuda.max_memory_allocated(0) / 1e9
print(f"\n[VRAM] Peak during training : {vram_peak:.2f} GB / {GPU_VRAM:.1f} GB")

Converting train dataset to ChatML:   0%|          | 0/17784 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/17784 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/17784 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/17784 [00:00<?, ? examples/s]

Converting eval dataset to ChatML:   0%|          | 0/937 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/937 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/937 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


[VRAM] Allocated  : 15.39 GB
[VRAM] Reserved   : 15.44 GB
[VRAM] Headroom   : 35.31 GB

  TRAINING START — STANDARD LoRA (BF16)
  Batch size (per device) : 2
  Gradient accum steps    : 8
  Effective batch size    : 16
  Max sequence length     : 2048
  CPU workers (70%)       : 33 of 48 cores
  Precision               : BF16
  Optimizer               : OptimizerNames.ADAMW_TORCH_FUSED
  Flash Attention         : sdpa


Step,Training Loss,Validation Loss
200,0.893900,0.914440


In [8]:
print("USE_FP16 :", USE_FP16)
print("USE_BF16 :", USE_BF16)
print("COMPUTE_DTYPE :", COMPUTE_DTYPE)
print("sft_config.fp16 :", sft_config.fp16)
print("sft_config.bf16 :", sft_config.bf16)

USE_FP16 : True
USE_BF16 : False
COMPUTE_DTYPE : torch.float16
sft_config.fp16 : True
sft_config.bf16 : False


In [15]:
print("Batch Size:", sft_config.per_device_train_batch_size)
print("Grad Accum:", sft_config.gradient_accumulation_steps)
print("Max Length:", sft_config.max_length)

Batch Size: 4
Grad Accum: 4
Max Length: 2048


In [9]:
# ============================================================
#  CELL 8: SAVE LoRA ADAPTERS + TOKENIZER
# ============================================================

SAVE_PATH = "./gurukul-llama3-lora-adapters"

trainer.model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"[SAVED] LoRA adapters -> {SAVE_PATH}")
print("\nSaved files:")
for f in sorted(os.listdir(SAVE_PATH)):
    size_mb = os.path.getsize(os.path.join(SAVE_PATH, f)) / 1e6
    print(f"  {f:45s} {size_mb:.1f} MB")

[SAVED] LoRA adapters -> ./gurukul-llama3-lora-adapters

Saved files:
  README.md                                     0.0 MB
  adapter_config.json                           0.0 MB
  adapter_model.safetensors                     167.8 MB
  special_tokens_map.json                       0.0 MB
  tokenizer.json                                17.2 MB
  tokenizer_config.json                         0.1 MB


In [10]:
# ============================================================
#  CELL 9: MERGE LoRA ADAPTERS INTO BASE MODEL
# ============================================================

from peft import PeftModel

MERGED_PATH = "./gurukul-llama3-lora-merged"

print("[INFO] Merging LoRA adapters into base model...")

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=COMPUTE_DTYPE,
    device_map="auto",
    trust_remote_code=True,
)

merged_model = PeftModel.from_pretrained(base_model, SAVE_PATH)
merged_model = merged_model.merge_and_unload()

merged_model.save_pretrained(MERGED_PATH, safe_serialization=True)
tokenizer.save_pretrained(MERGED_PATH)

print(f"[SAVED] Merged model -> {MERGED_PATH}")
print("\nMerged model files:")
for f in sorted(os.listdir(MERGED_PATH)):
    size_mb = os.path.getsize(os.path.join(MERGED_PATH, f)) / 1e6
    print(f"  {f:45s} {size_mb:.1f} MB")

[INFO] Merging LoRA adapters into base model...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
/home/poornima/miniconda3/envs/llama/lib/python3.11/site-packages/transformers/modeling_utils.py:3391: UserWarning: Attempting to save a model with offloaded modules. Ensure that unallocated cpu memory exceeds the `shard_size` (5GB default)
  warnings.warn(


Saving checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

[SAVED] Merged model -> ./gurukul-llama3-lora-merged

Merged model files:
  config.json                                   0.0 MB
  generation_config.json                        0.0 MB
  model-00001-of-00004.safetensors              4976.7 MB
  model-00002-of-00004.safetensors              4999.8 MB
  model-00003-of-00004.safetensors              4915.9 MB
  model-00004-of-00004.safetensors              1168.1 MB
  model.safetensors.index.json                  0.0 MB
  special_tokens_map.json                       0.0 MB
  tokenizer.json                                17.2 MB
  tokenizer_config.json                         0.1 MB


In [16]:
# ============================================================
#  CELL 10: INFERENCE TEST — VERIFY FINE-TUNING QUALITY
# ============================================================

from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=trainer.model,
    tokenizer=tokenizer,
    torch_dtype=COMPUTE_DTYPE,
    device_map="auto",
)

test_prompt = (
    "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
    "You are Gurukul, an AI educational assistant specialized in generating high-quality multiple-choice questions.\n"
    "<|eot_id|><|start_header_id|>user<|end_header_id|>\n"
    "Generate 5 multiple-choice questions from the following topic.\n\n"
    "Topic: Motion\n\n"
    "For each question provide:\n"
    "1. Question\n"
    "2. Four options (A, B, C, D)\n"
    "3. Correct Answer\n"
    "4. Brief Explanation\n"
    "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"
)
output = pipe(
    test_prompt,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.1,
)

print("\n[INFERENCE TEST OUTPUT]")
print("="*60)
generated = output[0]['generated_text']
response_start = generated.find("<|start_header_id|>assistant<|end_header_id|>")
if response_start != -1:
    print(generated[response_start + len("<|start_header_id|>assistant<|end_header_id|>"):])
else:
    print(generated)

Device set to use cuda:0
The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FalconMambaForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'Gemma2ForCausalLM', 'Gemma3ForConditionalGeneration', 'Gemma3ForCausalLM', 'GitForCausalLM', 'GlmForCausalLM', 'Glm4ForCausalLM', 'GotOcr2ForConditionalGeneration', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoFo


[INFERENCE TEST OUTPUT]

Question 1: Which of the following is not a vector quantity?
 A. Acceleration B. Angular velocity C. Displacement D. Time

Correct Answer: D. Time

Explanation: Time is a scalar quantity as it does not have both magnitude and direction.

Question 2: If the displacement of a particle moving in a plane at time t is given by x = a sin ωt + b cos ωt, then what is its amplitude?
 A. √(a² - b²) B. √(a² + b²) C. √(b² - a²) D. None of these

Correct Answer: B. √(a² + b²)

Explanation: The amplitude of this oscillatory motion is √(a² + b²).

Question 3: What is the net acceleration of an object that moves with uniform speed along a curved path?
 A. Zero B. Centripetal acceleration C. Tangential acceleration D. Uniform acceleration




In [11]:
from huggingface_hub import HfApi

repo_id = "maxbraintech/gurukul-llama3-merged"

api = HfApi()

api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    exist_ok=True
)

api.upload_folder(
    folder_path="./gurukul-llama3-lora-merged",
    repo_id=repo_id,
    repo_type="model"
)

print(f"Uploaded: https://huggingface.co/{repo_id}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded: https://huggingface.co/maxbraintech/gurukul-llama3-merged


In [13]:
# ============================================================
#  CELL 11: GPU & CPU UTILIZATION CHECK
#
#  Run in a SEPARATE SSH terminal during training:
#    watch -n 1 nvidia-smi
#
#  Expected readings (standard LoRA, RTX 8000):
#    GPU-Util : 95-100%
#    Mem-Usage: 26-35 GB / 48 GB
#    Temp     : 70-85 C
# ============================================================

import subprocess

result = subprocess.run(
    ["nvidia-smi",
     "--query-gpu=name,utilization.gpu,memory.used,memory.total,temperature.gpu",
     "--format=csv,noheader,nounits"],
    capture_output=True, text=True
)

print("[GPU STATUS]")
for line in result.stdout.strip().split('\n'):
    parts = [p.strip() for p in line.split(',')]
    if len(parts) == 5:
        print(f"  Device   : {parts[0]}")
        print(f"  GPU-Util : {parts[1]}%")
        print(f"  Memory   : {parts[2]} MB / {parts[3]} MB")
        print(f"  Temp     : {parts[4]} C")

print("\n[PYTORCH MEMORY SUMMARY]")
print(torch.cuda.memory_summary(device=0, abbreviated=True))

[GPU STATUS]
  Device   : Quadro RTX 8000
  GPU-Util : 33%
  Memory   : 33139 MB / 49152 MB
  Temp     : 30 C

[PYTORCH MEMORY SUMMARY]
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |  31129 MiB |  36433 MiB |   2051 TiB |   2051 TiB |
|---------------------------------------------------------------------------|
| Active memory         |  31129 MiB |  36433 MiB |   2051 TiB |   2051 TiB |
|---------------------------------------------------------------------------|
| Requ